Data is from [here](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/about_data)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jameskelleher/workshops/blob/main/data-analysis/notebook.ipynb)

In [26]:
import pandas as pd
import seaborn as sns
import sqlite3

from pprint import pprint

%pip install jupysql

In [3]:
url = 'https://github.com/jameskelleher/workshops/raw/refs/heads/main/data-analysis/311-service-requests.xlsx'
df = pd.read_excel(url, sheet_name='All Agencies Complaint<>Details', skiprows=10)
df.head(10)

,Agency,Complaint Type,Descriptor,Additional_Details,Public View,Location_Type
0,Department for the Aging,Case Management Agency Complaint,NaN,NaN,No,Address Unknown
1,Department for the Aging,Case Management Agency Complaint,NaN,NaN,No,Senior Address
2,Department for the Aging,Elder Abuse,NaN,NaN,No,Senior Address
3,Department for the Aging,Eviction,Homebound Senior,NaN,No,Address Unknown
4,Department for the Aging,Eviction,Mobile Senior,NaN,No,Address Unknown
5,Department for the Aging,Eviction,Mobile Senior,NaN,No,Senior Address
6,Department for the Aging,Eviction,Homebound Senior,NaN,No,Senior Address
7,Department for the Aging,HEAP Assistance,NaN,NaN,No,Address Unknown
8,Department for the Aging,HEAP Assistance,NaN,NaN,No,Senior Address
9,Department for the Aging,Home Delivered Meal - Missed Delivery,NaN,NaN,No,Address Unknown


In [4]:
agencies = df['Agency'].unique()
sorted(agencies)

['Department for the Aging',
 'Department of Buildings',
 'Department of Consumer Affairs',
 'Department of Education',
 'Department of Environmental Protection',
 'Department of Finance',
 'Department of Health and Mental Hygiene',
 'Department of Homeless Services',
 'Department of Parks and Recreation',
 'Department of Sanitation',
 'Department of Transportation',
 'Economic Development Corporation',
 'Human Resources Administration',
 'NYC Emergency Management',
 'New York City Police Department',
 'Office of Technology and Innovation',
 'Taxi and Limousine Commission']

In [5]:
sanitation_rows = df['Agency'] == 'Department of Sanitation'
sanitation_rows

,Agency
0,False
1,False
2,False
3,False
4,False
...,...
5958,False
5959,False
5960,False
5961,False


In [6]:
print(f'type: {type(sanitation_rows)}')
print(f'dtype: {sanitation_rows.dtype}')
print(f'count: {sanitation_rows.sum()}')

pct = sanitation_rows.sum() / sanitation_rows.count() * 100
print(f'pct: {pct:.2f}%')

type: <class 'pandas.core.series.Series'>
dtype: bool
count: 278
pct: 4.66%


In [7]:
san_df = df[sanitation_rows]
san_df.head()

,Agency,Complaint Type,Descriptor,Additional_Details,Public View,Location_Type
4487,Department of Sanitation,Abandoned Bike,Chained to Public Property,NaN,Yes,Intersection
4488,Department of Sanitation,Abandoned Bike,Chained to Public Property,NaN,Yes,Street
4489,Department of Sanitation,Abandoned Vehicle,Without License Plate,NaN,Yes,Street
4490,Department of Sanitation,Commercial Disposal Complaint,No Private Carter,NaN,Yes,Street
4491,Department of Sanitation,Commercial Disposal Complaint,Private Carter Decal Not Posted,NaN,Yes,Street


In [8]:
def col_contains(df, col, word):
    return df[col].str.contains(word, case=False, na=False)

mask = (
    col_contains(san_df, 'Additional_Details', 'dog') |
    col_contains(san_df, 'Complaint Type', 'dog') | 
    col_contains(san_df, 'Descriptor', 'dog')
)

san_df[mask]

,Agency,Complaint Type,Descriptor,Additional_Details,Public View,Location_Type
4505,Department of Sanitation,Dead Animal,Dog,Highway,Yes,Highway
4506,Department of Sanitation,Dead Animal,Dog,Street or Sidewalk,Yes,Sidewalk
4507,Department of Sanitation,Dead Animal,Dog,Street or Sidewalk,Yes,Street
4533,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Sidewalk
4534,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Street
4535,Department of Sanitation,Dirty Condition,Dog Waste,Not Picked Up by Dog Owner,Yes,Sidewalk
4536,Department of Sanitation,Dirty Condition,Dog Waste,Not Picked Up by Dog Owner,Yes,Street


In [14]:
def make_mask(df, cols, words):
    if isinstance(cols, str):
        cols = [cols]
        
    if isinstance(words, str):
        words = [words]

    mask = pd.Series(False, index=df.index)
    for col in cols:
        for word in words:
            mask = mask | df[col].str.contains(word, case=False, na=False)
    return mask

cols = ['Complaint Type', 'Descriptor', 'Additional_Details']
dog_mask = make_mask(df, cols, 'dog')

df[dog_mask]

,Agency,Complaint Type,Descriptor,Additional_Details,Public View,Location_Type
612,Department of Environmental Protection,Noise,Animal Other Than Dog,NaN,Yes,Commercial/Mixed Use Building
613,Department of Environmental Protection,Noise,Animal Other Than Dog,NaN,Yes,Residential
621,Department of Environmental Protection,Noise,Dog,NaN,Yes,Commercial/Mixed Use Building
622,Department of Environmental Protection,Noise,Dog,NaN,Yes,Residential
2909,Department of Health and Mental Hygiene,Pet Shop,Dogs or Cats Not Sold,Care of Animals,Yes,Commercial Building
...,...,...,...,...,...,...
5464,New York City Police Department,Animal-Abuse,Tortured,Dog,Yes,Parking Lot
5471,New York City Police Department,Animal-Abuse,Tortured,Dog,Yes,Residential Building/House
5474,New York City Police Department,Animal-Abuse,Tortured,Dog,Yes,Store/Commercial
5481,New York City Police Department,Animal-Abuse,Tortured,Dog,Yes,Street/Sidewalk


In [10]:
waste_mask = make_mask(df, cols, 'waste')
mask = dog_mask & waste_mask

print(len(df[mask]))
df[mask]

11


,Agency,Complaint Type,Descriptor,Additional_Details,Public View,Location_Type
3770,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,1-2 Family Dwelling
3771,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,1-2 Family Mixed Use Building
3774,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,3+ Family Apartment Building
3775,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,3+ Family Mixed Use Building
3778,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,Commercial Building
3904,Department of Parks and Recreation,Animal in a Park,Animal Waste,Dog,Yes,Park
3907,Department of Parks and Recreation,Animal in a Park,Animal Waste,Dog,Yes,Street/Curbside
4533,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Sidewalk
4534,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Street
4535,Department of Sanitation,Dirty Condition,Dog Waste,Not Picked Up by Dog Owner,Yes,Sidewalk


In [11]:
filter_out_words = ['stray', 'noise', 'shop', 'leash', 'license', 'abuse', 'dead', 'odor']
filter_out = make_mask(df, cols, filter_out_words)

mask = dog_mask & ~filter_out
print(len(df[mask]))
df[mask]

11


,Agency,Complaint Type,Descriptor,Additional_Details,Public View,Location_Type
3770,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,1-2 Family Dwelling
3771,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,1-2 Family Mixed Use Building
3774,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,3+ Family Apartment Building
3775,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,3+ Family Mixed Use Building
3778,Department of Health and Mental Hygiene,Unsanitary Animal Pvt Property,Dog,Waste,Yes,Commercial Building
3904,Department of Parks and Recreation,Animal in a Park,Animal Waste,Dog,Yes,Park
3907,Department of Parks and Recreation,Animal in a Park,Animal Waste,Dog,Yes,Street/Curbside
4533,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Sidewalk
4534,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Street
4535,Department of Sanitation,Dirty Condition,Dog Waste,Not Picked Up by Dog Owner,Yes,Sidewalk


In [18]:
agency_mask = make_mask(df, 'Agency', ['sanitation', 'recreation'])
mask = agency_mask & dog_mask & waste_mask
df[mask]

,Agency,Complaint Type,Descriptor,Additional_Details,Public View,Location_Type
3904,Department of Parks and Recreation,Animal in a Park,Animal Waste,Dog,Yes,Park
3907,Department of Parks and Recreation,Animal in a Park,Animal Waste,Dog,Yes,Street/Curbside
4533,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Sidewalk
4534,Department of Sanitation,Dirty Condition,Dog Waste,Not Cleaned by Property Owner,Yes,Street
4535,Department of Sanitation,Dirty Condition,Dog Waste,Not Picked Up by Dog Owner,Yes,Sidewalk
4536,Department of Sanitation,Dirty Condition,Dog Waste,Not Picked Up by Dog Owner,Yes,Street


In [20]:
taxis = sns.load_dataset('taxis')
taxis.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


In [33]:
conn = sqlite3.connect(':memory:')
taxis.to_sql('taxis', conn, if_exists='replace', index=False)

6433

In [34]:
%reload_ext sql
%sql conn

In [27]:
%load_ext sql
%sql conn

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [37]:
%%sql
SELECT *
FROM taxis

Running query in 'Connection'

pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.6,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.0,0.0,9.3,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.7,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.1,0.0,13.4,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan
2019-03-11 10:37:23,2019-03-11 10:47:31,1,0.49,7.5,2.16,0.0,12.96,yellow,credit card,Times Sq/Theatre District,Midtown East,Manhattan,Manhattan
2019-03-26 21:07:31,2019-03-26 21:17:29,1,3.65,13.0,2.0,0.0,18.8,yellow,credit card,Battery Park City,Two Bridges/Seward Park,Manhattan,Manhattan
2019-03-22 12:47:13,2019-03-22 12:58:17,0,1.4,8.5,0.0,0.0,11.8,yellow,None,Murray Hill,Flatiron,Manhattan,Manhattan
2019-03-23 11:48:50,2019-03-23 12:06:14,1,3.63,15.0,1.0,0.0,19.3,yellow,credit card,East Harlem South,Midtown Center,Manhattan,Manhattan
2019-03-08 16:18:37,2019-03-08 16:26:57,1,1.52,8.0,1.0,0.0,13.3,yellow,credit card,Lincoln Square East,Central Park,Manhattan,Manhattan


In [ ]:
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=500"
df = pd.read_csv(url)

In [ ]:
df2.head()